# Thu thập dữ liệu batdongsan.vn
**Phụ trách:** Nguyễn Bá Công · **Chạy:** 31/08–01/09/2026 · robots.txt kiểm tra 01/09/2026

- Không cần chạy lại để tái lập kết quả: dữ liệu thô đã lưu sẵn ở `data/raw/`.
- Crawler ghi ra `batdongsanVN.csv` trong thư mục đang chạy; nhóm đổi tên thành `data/raw/bat_dong_san_raw.xls` trước khi làm sạch.
- Có checkpoint mỗi 100 tin và nghỉ ngẫu nhiên giữa các request để giảm tải cho máy chủ.
- Chạy lại toàn bộ mất nhiều giờ và phụ thuộc giao diện trang tại thời điểm chạy, nên số tin có thể khác.

In [1]:
!pip install -q requests beautifulsoup4 lxml pandas openpyxl tqdm

In [ ]:
# ============================================================
# BATDONGSAN.VN CRAWLER
# ============================================================

import os
import re
import json
import time
import random
import threading
import requests
import pandas as pd

from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm


# ============================================================
# 1. CONFIG
# ============================================================

ROW = 50000

START_PAGE = 1

BASE_URL = "https://batdongsan.vn/ban-nha-dat/p{}"

CHECKPOINT_FILE = "checkpoint_batdongsanVN.csv"
STATE_FILE = "checkpoint_batdongsanVN_state.json"

OUTPUT_CSV = "batdongsanVN.csv"
OUTPUT_XLSX = "batdongsanVN.xlsx"


COLUMNS = [
    "Tieu_de",
    "Gia_ban",
    "Dien_tich",
    "Dia_diem",
    "Loai_hinh_BDS",
    "Mo_ta_Dac_diem",
    "Nguoi_dang_Chu_dau_tu",
    "Ngay_dang",
    "URL"
]


MAX_WORKERS = 12

RETRIES = 3

TIMEOUT = 20

CHECKPOINT_EVERY = 100


# ============================================================
# 2. SESSION
# ============================================================

_thread_local = threading.local()


def get_session():

    if not hasattr(
        _thread_local,
        "session"
    ):

        session = requests.Session()

        retry_strategy = Retry(
            total=RETRIES,
            connect=RETRIES,
            read=RETRIES,
            status=RETRIES,
            backoff_factor=0.5,

            status_forcelist=[
                429,
                500,
                502,
                503,
                504
            ],

            allowed_methods=[
                "GET"
            ]
        )

        adapter = HTTPAdapter(
            max_retries=retry_strategy,
            pool_connections=MAX_WORKERS,
            pool_maxsize=MAX_WORKERS
        )

        session.mount(
            "https://",
            adapter
        )

        session.mount(
            "http://",
            adapter
        )

        session.headers.update({

            "User-Agent":
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/151.0.0.0 Safari/537.36",

            "Accept-Language":
                "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",

            "Accept":
                "text/html,application/xhtml+xml,"
                "application/xml;q=0.9,image/avif,image/webp,"
                "*/*;q=0.8"
        })

        _thread_local.session = session

    return _thread_local.session


# ============================================================
# 3. GET HTML
# ============================================================

def get_soup(url):

    try:

        response = get_session().get(
            url,
            timeout=TIMEOUT
        )

        if response.status_code != 200:
            return None

        return BeautifulSoup(
            response.content,
            "lxml"
        )

    except Exception:
        return None


# ============================================================
# 4. CLEAN
# ============================================================

def clean_text(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace(
        "\xa0",
        " "
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def normalize_url(url):

    if not url:
        return ""

    url = url.strip()

    if url.startswith("/"):
        url = "https://batdongsan.vn" + url

    return url.split("#")[0]


# ============================================================
# 5. TITLE
# ============================================================

def extract_title(soup):

    if not soup:
        return ""

    # DOM chính xác của trang chi tiết
    h1 = soup.select_one(".content h1")

    if h1:
        value = clean_text(
            h1.get_text(" ", strip=True)
        )

        if value:
            return value

    # Fallback nếu cấu trúc HTML thay đổi
    h1 = soup.find("h1")

    if h1:
        value = clean_text(
            h1.get_text(" ", strip=True)
        )

        if value:
            return value

    return ""


# ============================================================
# 6. TÌM VÙNG THÔNG TIN TIN ĐĂNG
#
# QUAN TRỌNG:
#
# Không quét toàn trang.
#
# Không lấy label ở thanh FILTER.
#
# Chỉ lấy phần nằm sau H1.
# ============================================================

def get_detail_region(soup):

    h1 = soup.find("h1")

    if not h1:
        return soup

    parent = h1.parent

    if not parent:
        return soup

    # Ưu tiên container lớn chứa H1
    for _ in range(5):

        text = clean_text(
            parent.get_text(
                " ",
                strip=True
            )
        )

        if (
            "Mức giá" in text
            and "Diện tích" in text
            and len(text) < 30000
        ):
            return parent

        parent = parent.parent

        if not parent:
            break

    return soup


# ============================================================
# 7. LẤY GIÁ TRONG VÙNG DETAIL
# ============================================================

def extract_price(soup):

    if not soup:
        return ""

    for label in soup.select(".label"):

        label_text = clean_text(
            label.get_text(" ", strip=True)
        )

        if label_text.lower() != "mức giá":
            continue

        parent = label.parent

        if not parent:
            continue

        value = parent.select_one(".value")

        if value:

            result = clean_text(
                value.get_text(" ", strip=True)
            )

            if result:
                return result

    return ""
# ============================================================
# 8. DIỆN TÍCH
# ============================================================

def extract_area(soup):

    if not soup:
        return ""

    for label in soup.select(".label"):

        label_text = clean_text(
            label.get_text(" ", strip=True)
        )

        if label_text.lower() != "diện tích":
            continue

        parent = label.parent

        if not parent:
            continue

        value = parent.select_one(".value")

        if value:

            result = clean_text(
                value.get_text(" ", strip=True)
            )

            if result:
                return result

    return ""
# ============================================================
# 9. ĐỊA ĐIỂM
#
# CHỈ GIỮ TỈNH / THÀNH PHỐ
# ============================================================

PROVINCES = [

    "An Giang",
    "Bà Rịa - Vũng Tàu",
    "Bắc Giang",
    "Bắc Kạn",
    "Bạc Liêu",
    "Bắc Ninh",
    "Bến Tre",
    "Bình Định",
    "Bình Dương",
    "Bình Phước",
    "Bình Thuận",
    "Cà Mau",
    "Cao Bằng",
    "Cần Thơ",
    "Đà Nẵng",
    "Đắk Lắk",
    "Đắk Nông",
    "Điện Biên",
    "Đồng Nai",
    "Đồng Tháp",
    "Gia Lai",
    "Hà Giang",
    "Hà Nam",
    "Hà Nội",
    "Hà Tĩnh",
    "Hải Dương",
    "Hải Phòng",
    "Hậu Giang",
    "Hòa Bình",
    "Hưng Yên",
    "Khánh Hòa",
    "Kiên Giang",
    "Kon Tum",
    "Lai Châu",
    "Lâm Đồng",
    "Lạng Sơn",
    "Lào Cai",
    "Long An",
    "Nam Định",
    "Nghệ An",
    "Ninh Bình",
    "Ninh Thuận",
    "Phú Thọ",
    "Phú Yên",
    "Quảng Bình",
    "Quảng Nam",
    "Quảng Ngãi",
    "Quảng Ninh",
    "Quảng Trị",
    "Sóc Trăng",
    "Sơn La",
    "Tây Ninh",
    "Thái Bình",
    "Thái Nguyên",
    "Thanh Hóa",
    "Thừa Thiên Huế",
    "Tiền Giang",
    "TP. Hồ Chí Minh",
    "Hồ Chí Minh",
    "Trà Vinh",
    "Tuyên Quang",
    "Vĩnh Long",
    "Vĩnh Phúc",
    "Yên Bái"
]


def normalize_province(location):

    if not location:
        return ""

    location = clean_text(
        location
    )

    lower = location.lower()

    # --------------------------------------------------------
    # TP HỒ CHÍ MINH
    # --------------------------------------------------------

    hcm_patterns = [
        "tp. hồ chí minh",
        "tp hồ chí minh",
        "tp.hồ chí minh",
        "thành phố hồ chí minh",
        "hồ chí minh",
        "tphcm",
        "tp hcm",
        "tp.hcm",
        "sài gòn",
        "saigon"
    ]

    for pattern in hcm_patterns:

        if pattern in lower:

            return "TP. Hồ Chí Minh"

    # --------------------------------------------------------
    # Các tỉnh/thành khác
    # --------------------------------------------------------

    # sắp xếp tên dài trước
    sorted_provinces = sorted(
        PROVINCES,
        key=len,
        reverse=True
    )

    for province in sorted_provinces:

        if province.lower() in lower:

            return province

    return ""


def extract_raw_location(soup):

    h1 = soup.find("h1")

    if not h1:
        return ""

    # --------------------------------------------------------
    # Chỉ tìm trong các phần tử gần H1
    # --------------------------------------------------------

    current = h1

    for _ in range(15):

        current = current.find_next()

        if not current:
            break

        text = clean_text(
            current.get_text(
                " ",
                strip=True
            )
        )

        if not text:
            continue

        if len(text) > 250:
            continue

        lower = text.lower()

        # bỏ label
        if lower in [
            "mức giá",
            "diện tích",
            "phòng ngủ",
            "thông tin mô tả",
            "đặc điểm bất động sản"
        ]:
            continue

        # phải có dấu hiệu địa điểm
        if (
            "," in text
            or "phường" in lower
            or "xã " in lower
            or "quận" in lower
            or "huyện" in lower
            or "thành phố" in lower
            or "tp." in lower
            or "tỉnh " in lower
        ):

            province = normalize_province(
                text
            )

            if province:
                return province

    # --------------------------------------------------------
    # Fallback:
    # lấy text đầu trang sau H1
    # --------------------------------------------------------

    text = clean_text(
        soup.get_text(
            " ",
            strip=True
        )
    )

    # chỉ lấy đoạn trước "Mức giá"
    match = re.search(
        r"<h1>.*?</h1>\s*(.*?)\s+Mức giá",
        str(soup),
        flags=re.I | re.S
    )

    if match:

        candidate = clean_text(
            BeautifulSoup(
                match.group(1),
                "lxml"
            ).get_text(
                " ",
                strip=True
            )
        )

        province = normalize_province(
            candidate
        )

        if province:
            return province

    # tìm tỉnh/thành trong 1000 ký tự đầu
    first_part = text[:1500]

    return normalize_province(
        first_part
    )


# ============================================================
# 10. LOẠI HÌNH BĐS
# ============================================================

def extract_property_type(
    soup,
    title,
    description
):

    title_lower = clean_text(
        title
    ).lower()

    description_lower = clean_text(
        description
    ).lower()

    # --------------------------------------------------------
    # URL
    # --------------------------------------------------------

    canonical = soup.find(
        "link",
        rel="canonical"
    )

    url = ""

    if canonical:
        url = canonical.get(
            "href",
            ""
        ).lower()

    # --------------------------------------------------------
    # URL
    # --------------------------------------------------------

    url_patterns = [

        ("biet-thu", "Biệt thự"),
        ("villa", "Biệt thự"),

        ("shophouse", "Shophouse"),

        ("can-ho", "Căn hộ"),
        ("chung-cu", "Căn hộ"),

        ("nha-mat-pho", "Nhà mặt phố"),
        ("nha-mat-tien", "Nhà mặt tiền"),

        ("nha-pho", "Nhà phố"),

        ("nha-rieng", "Nhà riêng"),

        ("dat-nen", "Đất nền"),

        ("dat-tho-cu", "Đất thổ cư"),

        ("dat", "Đất")
    ]

    for keyword, result in url_patterns:

        if keyword in url:
            return result

    # --------------------------------------------------------
    # TITLE
    # --------------------------------------------------------

    title_patterns = [

        ("biệt thự", "Biệt thự"),
        ("villa", "Biệt thự"),

        ("shophouse", "Shophouse"),

        ("căn hộ", "Căn hộ"),
        ("chung cư", "Căn hộ"),

        ("nhà mặt phố", "Nhà mặt phố"),
        ("nhà mặt tiền", "Nhà mặt tiền"),

        ("nhà phố", "Nhà phố"),

        ("nhà riêng", "Nhà riêng"),

        ("đất nền", "Đất nền"),

        ("đất thổ cư", "Đất thổ cư")
    ]

    for keyword, result in title_patterns:

        if keyword in title_lower:
            return result

    # --------------------------------------------------------
    # DESCRIPTION
    # --------------------------------------------------------

    description_patterns = [

        ("biệt thự", "Biệt thự"),
        ("villa", "Biệt thự"),

        ("shophouse", "Shophouse"),

        ("căn hộ", "Căn hộ"),
        ("chung cư", "Căn hộ"),

        ("nhà mặt phố", "Nhà mặt phố"),
        ("nhà mặt tiền", "Nhà mặt tiền"),

        ("nhà phố", "Nhà phố"),

        ("nhà riêng", "Nhà riêng"),

        ("đất nền", "Đất nền"),
        ("đất thổ cư", "Đất thổ cư")
    ]

    for keyword, result in description_patterns:

        if keyword in description_lower:
            return result

    # --------------------------------------------------------
    # TITLE CHỈ CÓ "BÁN NHÀ"
    # --------------------------------------------------------

    if (
        re.search(
            r"\bbán nhà\b",
            title_lower
        )
        or
        re.search(
            r"\bmua bán nhà\b",
            title_lower
        )
    ):

        return "Nhà"

    if "bán đất" in title_lower:
        return "Đất"

    return ""


# ============================================================
# 11. MÔ TẢ
# ============================================================

def extract_description(soup):

    if not soup:
        return ""

    parts = []

    # Phần mô tả chính
    more = soup.find(id="more")

    if more:

        text = clean_text(
            more.get_text(
                " ",
                strip=True
            )
        )

        if text:
            parts.append(text)

    # Phần nội dung sau "Xem thêm"
    more1 = soup.find(id="more1")

    if more1:

        text = clean_text(
            more1.get_text(
                " ",
                strip=True
            )
        )

        if text:
            parts.append(text)

    # Ghép 2 phần lại
    description = clean_text(
        " ".join(parts)
    )

    return description[:12000]


# ============================================================
# 12. NGƯỜI ĐĂNG
# ============================================================

def extract_owner(soup):

    if not soup:
        return ""

    owner = soup.select_one(".profile-element .name")

    if owner:
        return clean_text(
            owner.get_text(" ", strip=True)
        )

    return ""


# ============================================================
# 13. NGÀY ĐĂNG
# ============================================================

def extract_post_date(soup):

    if not soup:
        return ""

    # Tìm đúng label "Ngày đăng"
    for label in soup.select(".label"):

        label_text = clean_text(
            label.get_text(" ", strip=True)
        )

        if label_text.lower() != "ngày đăng":
            continue

        # Lấy value nằm cùng block với label
        parent = label.parent

        if not parent:
            continue

        value = parent.select_one(".value")

        if value:

            result = clean_text(
                value.get_text(" ", strip=True)
            )

            # Kiểm tra đúng định dạng ngày
            if re.fullmatch(
                r"\d{1,2}/\d{1,2}/\d{4}",
                result
            ):
                return result

    return ""


# ============================================================
# 14. SCRAPE DETAIL
# ============================================================

def scrape_detail(url):

    soup = get_soup(
        url
    )

    if soup is None:
        return None

    try:

        title = extract_title(
            soup
        )

        if not title:
            return None

        price = extract_price(
            soup
        )

        area = extract_area(
            soup
        )

        location = extract_raw_location(
            soup
        )

        description = extract_description(
            soup
        )

        property_type = extract_property_type(
            soup,
            title,
            description
        )

        owner = extract_owner(
            soup
        )

        post_date = extract_post_date(
            soup
        )

        return {

            "Tieu_de":
                title,

            "Gia_ban":
                price,

            "Dien_tich":
                area,

            "Dia_diem":
                location,

            "Loai_hinh_BDS":
                property_type,

            "Mo_ta_Dac_diem":
                description,

            "Nguoi_dang_Chu_dau_tu":
                owner,

            "Ngay_dang":
                post_date,

            "URL":
                url
        }

    except Exception:

        return None


# ============================================================
# 15. GET DETAIL URLS
# ============================================================

def get_detail_urls(page):

    url = BASE_URL.format(
        page
    )

    soup = get_soup(
        url
    )

    if soup is None:
        return []

    urls = []

    seen = set()

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = normalize_url(
            a.get("href")
        )

        if not href:
            continue

        # Chỉ URL tin rao
        if not re.search(
            r"-r\d+$",
            href.rstrip("/")
        ):
            continue

        if "batdongsan.vn" not in href:
            continue

        if href in seen:
            continue

        seen.add(
            href
        )

        urls.append(
            href
        )

    return urls


# ============================================================
# 16. SAVE CHECKPOINT
# ============================================================

def save_checkpoint(
    rows,
    current_page
):

    if not rows:
        return

    df = pd.DataFrame(
        rows,
        columns=COLUMNS
    )

    df = df.fillna("")

    df = df.drop_duplicates(
        subset=["URL"],
        keep="first"
    )

    df.to_csv(
        CHECKPOINT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    state = {

        "current_page":
            current_page,

        "rows":
            len(df),

        "last_saved":
            time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }

    with open(
        STATE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            state,
            f,
            ensure_ascii=False,
            indent=2
        )


# ============================================================
# 17. LOAD CHECKPOINT
# ============================================================

def load_checkpoint():

    if not os.path.exists(
        CHECKPOINT_FILE
    ):

        return [], set(), START_PAGE

    try:

        df = pd.read_csv(
            CHECKPOINT_FILE,
            encoding="utf-8-sig",
            dtype=str
        )

        df = df.fillna("")

        for col in COLUMNS:

            if col not in df.columns:
                df[col] = ""

        df = df[
            COLUMNS
        ]

        df = df.drop_duplicates(
            subset=["URL"],
            keep="first"
        )

        rows = df.to_dict(
            orient="records"
        )

        urls = set(
            df["URL"].tolist()
        )

        current_page = START_PAGE

        if os.path.exists(
            STATE_FILE
        ):

            try:

                with open(
                    STATE_FILE,
                    "r",
                    encoding="utf-8"
                ) as f:

                    state = json.load(
                        f
                    )

                current_page = int(
                    state.get(
                        "current_page",
                        START_PAGE
                    )
                )

            except Exception:
                pass

        return (
            rows,
            urls,
            current_page
        )

    except Exception:

        return [], set(), START_PAGE


# ============================================================
# 18. VALIDATE
# ============================================================

def validate_row(row):

    if not row:
        return False

    if not row.get(
        "URL",
        ""
    ):
        return False

    if not row.get(
        "Tieu_de",
        ""
    ):
        return False

    return True


# ============================================================
# 19. CRAWLER
# ============================================================

def crawl_batdongsan():

    rows, scraped_urls, current_page = (
        load_checkpoint()
    )

    print(
        "=" * 70
    )

    print(
        "BATDONGSAN.VN CRAWLER"
    )

    print(
        "=" * 70
    )

    print(
        f"Mục tiêu       : {ROW:,}"
    )

    print(
        f"Checkpoint      : {len(rows):,}"
    )

    print(
        f"Bắt đầu page    : {current_page}"
    )

    print(
        f"Workers         : {MAX_WORKERS}"
    )

    print(
        "=" * 70
    )

    last_saved = len(
        rows
    )

    try:

        with ThreadPoolExecutor(
            max_workers=MAX_WORKERS
        ) as executor:

            while len(rows) < ROW:

                print(
                    f"\nPAGE {current_page}"
                )

                page_urls = get_detail_urls(
                    current_page
                )

                print(
                    f"URL tìm thấy: {len(page_urls)}"
                )

                if not page_urls:

                    current_page += 1

                    continue

                new_urls = [

                    url

                    for url in page_urls

                    if url not in scraped_urls
                ]

                print(
                    f"URL mới: {len(new_urls)}"
                )

                if not new_urls:

                    current_page += 1

                    continue

                remaining = (
                    ROW
                    -
                    len(rows)
                )

                new_urls = new_urls[
                    :remaining
                ]

                futures = {

                    executor.submit(
                        scrape_detail,
                        url
                    ):
                    url

                    for url in new_urls
                }

                page_rows = []

                with tqdm(
                    total=len(futures),
                    desc=f"Page {current_page}",
                    unit="tin"
                ) as pbar:

                    for future in as_completed(
                        futures
                    ):

                        url = futures[
                            future
                        ]

                        try:

                            result = future.result()

                            if (
                                result
                                and
                                validate_row(
                                    result
                                )
                            ):

                                page_rows.append(
                                    result
                                )

                                scraped_urls.add(
                                    url
                                )

                        except Exception:
                            pass

                        pbar.update(
                            1
                        )

                rows.extend(
                    page_rows
                )

                # chống trùng
                unique = {}

                for row in rows:

                    url = row.get(
                        "URL",
                        ""
                    )

                    if url:
                        unique[url] = row

                rows = list(
                    unique.values()
                )

                rows = rows[
                    :ROW
                ]

                print(
                    f"Đã lấy: {len(rows):,}/{ROW:,}"
                )

                # ------------------------------------------------
                # SAVE
                # ------------------------------------------------

                if (
                    len(rows)
                    -
                    last_saved
                    >= CHECKPOINT_EVERY
                    or
                    page_rows
                ):

                    save_checkpoint(
                        rows,
                        current_page + 1
                    )

                    last_saved = len(
                        rows
                    )

                    print(
                        "✓ Đã lưu checkpoint"
                    )

                current_page += 1

                time.sleep(
                    random.uniform(
                        0.2,
                        0.5
                    )
                )

    except KeyboardInterrupt:

        print(
            "\n\nĐÃ DỪNG CRAWLER"
        )

        save_checkpoint(
            rows,
            current_page
        )

        print(
            f"Đã lưu {len(rows):,} dòng"
        )

        return pd.DataFrame(
            rows,
            columns=COLUMNS
        )

    except Exception as e:

        print(
            "\nCrawler lỗi:",
            repr(e)
        )

        save_checkpoint(
            rows,
            current_page
        )

        return pd.DataFrame(
            rows,
            columns=COLUMNS
        )

    # ========================================================
    # FINAL DATA
    # ========================================================

    df = pd.DataFrame(
        rows,
        columns=COLUMNS
    )

    df = df.fillna("")

    df = df.drop_duplicates(
        subset=["URL"],
        keep="first"
    )

    df = df.head(
        ROW
    )

    df = df.reset_index(
        drop=True
    )

    # ========================================================
    # SAVE CSV
    # ========================================================

    df.to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    # ========================================================
    # SAVE EXCEL
    # ========================================================

    df.to_excel(
        OUTPUT_XLSX,
        index=False
    )

    # ========================================================
    # CHECKPOINT
    # ========================================================

    save_checkpoint(
        df.to_dict(
            orient="records"
        ),
        current_page
    )

    print(
        "\n"
        + "=" * 70
    )

    print(
        "CRAWL HOÀN TẤT"
    )

    print(
        "=" * 70
    )

    print(
        f"Số dòng: {len(df):,}"
    )

    print(
        f"Số cột: {len(df.columns)}"
    )

    print(
        f"CSV: {OUTPUT_CSV}"
    )

    print(
        f"Excel: {OUTPUT_XLSX}"
    )

    print(
        f"Checkpoint: {CHECKPOINT_FILE}"
    )

    print(
        "\nTỷ lệ thiếu:"
    )

    print(
        df.isna().sum()
    )

    print(
        "\n10 dòng đầu:"
    )

    display(
        df.head(10)
    )

    return df


# ============================================================
# 20. START
# ============================================================

df = crawl_batdongsan()

BATDONGSAN.VN CRAWLER
Mục tiêu       : 50,000
Checkpoint      : 30,243
Bắt đầu page    : 14
Workers         : 12

PAGE 14
URL tìm thấy: 24
URL mới: 24


Page 14:   0%|          | 0/24 [00:00<?, ?tin/s]

Đã lấy: 30,267/50,000
✓ Đã lưu checkpoint

PAGE 15
URL tìm thấy: 24
URL mới: 21


Page 15:   0%|          | 0/21 [00:00<?, ?tin/s]